![title](../DOCS/img.png)

# SOAPGemma - Fine-tuning TxGemma for SOAP Note Generation

This notebook demonstrates the process of fine-tuning a Gemma model for generating medical SOAP notes from patient-doctor dialogues. It covers

* data loading
* model quantization
* LoRA-based fine-tuning
* inference.

# **What is SOAP ?** #
A SOAP note is a structured method of documentation used by healthcare providers. The acronym SOAP stands for:

* **S - Subjective**: This section captures information reported by the patient, such as their feelings, concerns, and their description of symptoms (e.g., "stomach pain," "nauseated, fatigued"). It includes the patient's chief complaint and the history of their present illness.

* **O - Objective**: This part includes observable, measurable, and factual data collected by the clinician. This encompasses vital signs (temperature, heart rate, blood pressure, etc.), physical exam findings, laboratory results, and imaging data.

* **A - Assessment**: Here, the clinician provides their professional judgment and diagnosis based on the subjective and objective information gathered. It involves an analysis of the patient's condition, potential diagnoses, and the patient's progress.

* **P - Plan**: This section outlines the treatment plan, including any further tests, therapies, medications, referrals to specialists, and follow-up actions.

SOAP notes are a crucial tool for healthcare workers to organize patient information, guide clinical reasoning, and facilitate communication among health professionals. They help ensure consistent and clear documentation, which is essential for quality patient care. This standardized format was developed by Dr. Lawrence Weed in the 1960s.

# What is TxGemma ? #

TxGemma is a collection of open-source machine learning models designed to improve the efficiency of therapeutic development. These models are fine-tuned from Google DeepMind's Gemma 2 architecture using a large dataset (7 million training examples) from the Therapeutics Data Commons (TDC), which includes information on small molecules, proteins, nucleic acids, diseases, and cell lines.

TxGemma models come in various sizes (2B, 9B, and 27B parameters) and are built to Predict therapeutic properties, Perform classification, regression, and generation tasks, Facilitate conversational AI for deeper insights, Support agentic orchestration.

_(Google, "TXGemma: A Family of Lightweight Open Models," Google AI Blog, accessed May 22, 2025, https://blog.google/technology/ai/gemma-open-models/.)_

# What is OMI ? #

OMI dataset consists of 10,000 synthetic dialogues between a patient and clinician, created using the GPT-4 dataset from NoteChat, based on PubMed Central (PMC) case-reports. Accompanying these dialogues are SOAP summaries generated through GPT-4. The dataset is split into 9250 training, 500 validation, and 250 test entries, each containing a dialogue column, a SOAP column, a prompt column, and a ChatML-style conversation format column.

_(Junxian Tang et al., "NoteChat: A Dataset of Synthetic Doctor-Patient Conversations Conditioned on Clinical Notes," arXiv, last modified October 24, 2023, https://arxiv.org/abs/2310.15959.)_

## 1. Setup and Imports

This cell imports necessary libraries, including `torch` for deep learning, `bitsandbytes` for quantization, `pathlib` for path management, `transformers` for accessing pre-trained models and tokenizers, and `datasets` for handling the data. It also defines paths for data and models and checks for CUDA availability to leverage GPU acceleration.

In [6]:
import torch
import bitsandbytes as bnb
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset

DATA_PATH = Path("../data")
OMI_PATH_processed = DATA_PATH / "processed" / "omi-health"
OMI_PATH_raw = DATA_PATH / "raw" / "omi-health"
MODEL_PATH =  Path("../models")

print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device name: {torch.cuda.get_device_name(0)}")

print(f"Bitsandbytes version: {bnb.__version__}")

C:\Users\khal6952\.conda\envs\SOAPG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch CUDA available: True
CUDA version: 12.9
CUDA device count: 1
Current CUDA device name: NVIDIA GeForce RTX 5070 Ti
Bitsandbytes version: 0.46.1


## 2. Load Model and Tokenizer

The `google/txgemma-2b-predict` model is loaded. To reduce memory usage, 4-bit quantization is applied using `BitsAndBytesConfig`. This involves loading the model in 4-bit precision (`load_in_4bit=True`), using the "nf4" quantization type (`bnb_4bit_quant_type="nf4"`), and specifying `torch.bfloat16` as the computation dtype (`bnb_4bit_compute_dtype=torch.bfloat16`). The tokenizer corresponding to the model is also loaded. The model is mapped to the available CUDA device.

In [7]:
model_id = "google/medgemma-4b-it"

# Use 4-bit quantization to reduce memory usage
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map={"":0},
    torch_dtype="auto",
    attn_implementation="eager",
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.68s/it]


## 3. Load and Prepare Dataset

### 3.1. Load Data from CSV
The training dataset is loaded from a CSV file (`train_v1.csv`) located in the processed data path. The `head()` method is used to display the first few rows of the DataFrame, showing columns like `dialogue`, `soap` (the target SOAP note), `prompt`, and `messages`.

In [8]:
import pandas as pd

# Load the dataset
train_df = pd.read_csv(OMI_PATH_processed / "train_v1.csv")
train_df.head()

,dialogue,soap,prompt,messages,messages_nosystem,event_tags
0,"Doctor: Hello, how can I help you today?\nPati...",S: The patient's mother reports that her 13-ye...,Create a Medical SOAP note summary from the di...,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'user', 'content': ""You are an exper...",['(After the tests)']
1,"Doctor: Hello, what brings you in today?\nPati...","S: The patient, a 21-month-old male, presented...",Create a Medical SOAP note summary from the di...,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'user', 'content': ""You are an exper...","['[After the tests]', '[After 3 weeks of thera..."
2,"Doctor: Hello, how can I help you today?\nPati...","S: Patient reports experiencing fatigue, night...",Create a Medical SOAP note summary from the di...,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'user', 'content': ""You are an exper...",[]
3,"Doctor: Hello, Patient D. How are you feeling ...","S: Patient D, a 60-year-old African American m...",Create a medical SOAP summary of this dialogue.,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'user', 'content': ""You are an exper...",[]
4,"Doctor: Hello, I see that you have a history o...","S: The patient, a married woman with a 7-year ...",Create a Medical SOAP note summary from the di...,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'user', 'content': ""You are an exper...",[]


### 3.2. Convert to Hugging Face Dataset
The pandas DataFrame is converted into a `Dataset` object from the `datasets` library. The 'soap' column is renamed to 'completion' as this is often the expected target column name for supervised fine-tuning tasks. Unnecessary columns (`prompt`, `messages`, `messages_nosystem`) are removed to simplify the dataset for training.

In [9]:
dataset = Dataset.from_pandas(train_df)
dataset = dataset.rename_column('soap', 'completion')
cols_to_drop = ['prompt', 'messages', 'messages_nosystem']
dataset = dataset.remove_columns(cols_to_drop)
dataset

Dataset({
    features: ['dialogue', 'completion', 'event_tags'],
    num_rows: 9250
})

## 4. Configure LoRA (Low-Rank Adaptation)

LoRA is a parameter-efficient fine-tuning (PEFT) technique. `LoraConfig` is used to set up the LoRA parameters:
- `r=8`: The rank of the LoRA matrices. A smaller rank means fewer trainable parameters.
- `task_type="CAUSAL_LM"`: Specifies that the task is Causal Language Modeling.
- `target_modules`: A list of the model's layers (specifically, attention and feed-forward network projections) to which LoRA will be applied.

In [10]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "o_proj",
        "k_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

## 5. Prepare Model for k-bit Training

The `prepare_model_for_kbit_training` function from PEFT preprocesses the quantized model for training. This step ensures that the model is compatible with k-bit training techniques. Then, `get_peft_model` applies the LoRA configuration to the quantized model, creating a `PeftModel` that is ready for fine-tuning.

In [ ]:
from peft import prepare_model_for_kbit_training, get_peft_model

# Preprocess quantized model for training
model = prepare_model_for_kbit_training(model)

# Create PeftModel from quantized model and configuration
model = get_peft_model(model, lora_config)

## 6. Define Formatting Function

This function, `format_dialogue_for_soap_synthesis_v2`, structures the input for the model. It takes a data sample (containing 'dialogue' and 'completion') and formats it into a single string: `"dialogue: {dialogue}<soap_start> soap_note:{soap} <eos>"`. This format clearly demarcates the input dialogue and the target SOAP note, with special tokens `<soap_start>` and `<eos>` (end of sequence) to guide the model during training and generation.

In [ ]:
def format_dialogue_for_soap_synthesis_v2(data):
    dialogue = data["dialogue"]
    soap = data["completion"]
    return f"dialogue: {dialogue}<soap_start> soap_note:{soap} <eos>"

## 7. Configure and Initialize Trainer

The `SFTTrainer` from the `trl` library is configured for fine-tuning the model. The critical hyperparameters set within `SFTConfig` that significantly impact the training process are:

* `learning_rate = 5e-4`: This is a crucial hyperparameter that dictates the step size at which the model updates its weights during training.
* `per_device_train_batch_size = 1` and `gradient_accumulation_steps = 4`: These parameters together define the effective batch size (1\*4 = 4). The batch size influences the gradient estimation and training stability.
* `max_steps = 100`: This determines the total number of training iterations the model will undergo.
* `max_seq_length = 2048`: Sets the maximum length of input sequences, affecting memory usage and the amount of context the model can consider.
* `fp16 = True`: Enables mixed-precision training. This can significantly speed up training and reduce memory footprint without a substantial loss in performance.
* `optim = "paged_adamw_8bit"`: Specifies the use of the paged AdamW optimizer in 8-bit, which is designed for more memory-efficient training of large models.

In [ ]:
import transformers
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=8,
        max_steps=100,
        learning_rate=5e-4,
        fp16=True,
        logging_steps=5,
        output_dir="/content/outputs",
        optim="paged_adamw_8bit",
        report_to="none",
    ),
    peft_config=lora_config,
    formatting_func=format_dialogue_for_soap_synthesis_v2,
)

## 8. Train the Model

The `trainer.train()` method starts the fine-tuning process. The output shows the training progress, including the current step, training loss, and epoch. The training loss generally decreases as the model learns from the data.

In [ ]:
trainer.train()

## 9. Save Model and Tokenizer

After training, the fine-tuned model (specifically, the LoRA adapters) and the tokenizer are saved to the directory specified by `MODEL_PATH / "SOAPgemma_v1"`. This allows for later reloading and use of the model without retraining.

In [2]:
# Save the fine-tuned model
model.save_pretrained(MODEL_PATH / "SOAPgemma_v2")

# Save the tokenizer
tokenizer.save_pretrained(MODEL_PATH / "SOAPgemma_v2")

print(f"Model and tokenizer saved to {MODEL_PATH / 'SOAPgemma_v2'}")

NameError: name 'model' is not defined

## 10. Define SOAP Note Generation Function

The `generate_soap_note` function takes a dialogue, the fine-tuned model, and the tokenizer as input to generate a SOAP note.
1.  **Input Formatting**: The input dialogue is formatted with the same prompt structure used during training: `"dialogue: {dialogue}<soap_start> soap_note:"`. This cues the model to generate the SOAP note part.
2.  **Tokenization**: The formatted input text is tokenized.
3.  **Generation**: The `model.generate()` method is called with the input IDs. Key parameters for generation include:
    * `max_new_tokens=512`: Limits the length of the generated SOAP note.
    * `num_beams=4`: Uses beam search with 4 beams to improve the quality of the generated text.
    * `temperature=0.7`: Controls the randomness of the generation. Lower values make the output more deterministic.
    * `pad_token_id=tokenizer.eos_token_id`: Sets the padding token ID, which is important for consistent generation.
4.  **Decoding**: The generated token IDs are decoded back into text. The input prompt part is sliced off from the output to return only the generated SOAP note.

In [11]:
def generate_soap_note(dialogue, model, tokenizer, device="cuda:0"):
    """Generates a SOAP note from a given dialogue."""

    # MODIFIED: Align with the training prompt structure to guide generation
    input_text = f"dialogue: {dialogue}<soap_start> soap_note:"

    # Tokenize the input
    inputs = tokenizer.encode_plus(
        input_text,
        return_tensors="pt",
        # Ensure tokenizer doesn't add EOS token here if the model adds it during generation
        add_special_tokens=True # Or False, depending on tokenizer and model behavior with this specific prompt
    ).to(device)

    # Generate the SOAP note
    outputs = model.generate(
        inputs.input_ids,
        # MODIFIED: Use max_new_tokens to control the length of the *generated* text
        max_new_tokens=512,  # Adjust as needed for typical SOAP note length
        num_beams=4,
        temperature=0.7,
        # IMPORTANT: Add pad_token_id if not set in model config, especially for open-ended generation
        pad_token_id=tokenizer.eos_token_id, # Or tokenizer.pad_token_id if different and model expects it
        # Consider adding an early stopping criterion or specific stop sequences if needed
        # eos_token_id=tokenizer.eos_token_id # Ensure model stops at <eos>
    )

    # Decode the output
    # MODIFIED: Slice the output to remove the input prompt text
    # The generated tokens start *after* the input_ids length
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    soap_note = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return soap_note

## 11. Test Generation

A sample dialogue is selected from the training data (`train_df.iloc[0,0]`). The `generate_soap_note` function is then called with this dialogue, the fine-tuned model, and the tokenizer to produce a SOAP note. The generated note is printed to the console.

In [12]:
sample_dialogue = train_df.iloc[0,0]
generated_note = generate_soap_note(sample_dialogue, model, tokenizer)
print("Generated SOAP Note:")
print(generated_note)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated SOAP Note:
 Patient: Hello, how can I help you today?
Doctor: My son has been having some issues with speech and development. He's 13 years old now.
Patient: I see. Can you tell me more about his symptoms? Does he have any issues with muscle tone or hypotonia?
Doctor: No, he doesn't have hypotonia. But he has mild to moderate speech and developmental delay, and he's been diagnosed with attention deficit disorder.
Patient: Thank you for sharing that information. We'll run some tests, including an MRI, to get a better understanding of your son's condition.
Doctor: Thank you for sharing that information. We'll run some tests, including an MRI, to get a better understanding of your son's condition.
(After the tests)
Doctor: The MRI results are in, and I'm glad to say that there are no structural brain anomalies. However, I did notice some physical characteristics. Does your son have any facial features like retrognathia, mild hypertelorism, or a slightly elongated philtrum and th

## 12. Display Original SOAP Note for Comparison

This cell displays the original SOAP note (`train_df.iloc[0,1]`) corresponding to the sample dialogue used for generation. This allows for a direct comparison between the model-generated SOAP note and the ground truth.

In [ ]:
train_df.iloc[0,1]